
# Capítulo 2. Transformación y modelado descriptivo de los datos de estratificación socioeconómica

## Metodología SEMMA: Modify y Model

Este notebook desarrolla las fases **Modify** y **Model** de la metodología SEMMA aplicadas al análisis de datos de estratificación socioeconómica en Colombia.

## Objetivos del capítulo

- Realizar limpieza y transformación de los datos.
- Clasificar los estratos socioeconómicos.
- Construir indicadores descriptivos.
- Elaborar tablas y gráficos comparativos.
- Desarrollar un modelo descriptivo territorial.
- Implementar dashboards interactivos.



# 2.1 Limpieza y depuración de los datos

En esta sección se realiza el proceso de transformación y depuración del conjunto de datos para garantizar consistencia y calidad durante el análisis descriptivo.


In [ ]:

# Librerías

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

pd.set_option('display.max_columns', None)


In [ ]:

# Carga del dataset procesado

df = pd.read_csv(
    r'C:\Users\STIVEN\OneDrive\universidad\diplomado\Final\data\processed\dataset_procesado.csv',
    sep=';',
    encoding='utf-8-sig'
)

df.head()


In [ ]:

# Información general del dataset

df.info()


In [ ]:

# Conversión de variables numéricas

df['area'] = (
    df['area']
    .astype(str)
    .str.replace(',', '.', regex=False)
)

df['perimetro'] = (
    df['perimetro']
    .astype(str)
    .str.replace(',', '.', regex=False)
)

df['area'] = pd.to_numeric(df['area'], errors='coerce')
df['perimetro'] = pd.to_numeric(df['perimetro'], errors='coerce')
df['numero_pisos'] = pd.to_numeric(df['numero_pisos'], errors='coerce')
df['estrato'] = pd.to_numeric(df['estrato'], errors='coerce')


In [ ]:

# Verificación de tipos de datos

df.dtypes


In [ ]:

# Valores faltantes

df.isnull().sum()


In [ ]:

# Eliminación de registros nulos

df = df.dropna()

print(df.shape)


In [ ]:

# Verificación de duplicados

df.duplicated().sum()


In [ ]:

# Eliminación de duplicados

df = df.drop_duplicates()

print(df.shape)


In [ ]:

# Estadísticas descriptivas

df.describe()


In [ ]:

# Visualización de valores atípicos

plt.figure(figsize=(10,5))

sns.boxplot(
    data=df[['area', 'perimetro']]
)

plt.title('Detección de valores atípicos')

plt.show()



# 2.2 Clasificación y agrupación de estratos socioeconómicos

Para facilitar el análisis descriptivo, los estratos socioeconómicos se agrupan en categorías generales de nivel bajo, medio y alto.


In [ ]:

# Clasificación de grupos socioeconómicos

def clasificar_estrato(x):

    if x in [1, 2]:
        return 'Bajo'

    elif x in [3, 4]:
        return 'Medio'

    else:
        return 'Alto'

df['grupo_estrato'] = df['estrato'].apply(clasificar_estrato)

df.head()


In [ ]:

# Distribución de grupos socioeconómicos

df['grupo_estrato'].value_counts()


In [ ]:

# Gráfico de grupos socioeconómicos

df['grupo_estrato'].value_counts().plot(
    kind='bar'
)

plt.xlabel('Grupo socioeconómico')
plt.ylabel('Cantidad')
plt.title('Distribución de grupos socioeconómicos')

plt.show()



# 2.3 Construcción de indicadores descriptivos

Se construyen indicadores descriptivos relacionados con área, cantidad de pisos y comportamiento de los estratos socioeconómicos.


In [ ]:

# Área promedio por estrato

indicador_area = df.groupby(
    'estrato'
)['area'].mean().round(2)

indicador_area


In [ ]:

# Promedio de pisos por estrato

indicador_pisos = df.groupby(
    'estrato'
)['numero_pisos'].mean().round(2)

indicador_pisos


In [ ]:

# Indicadores promedio por departamento

indicadores_departamento = df.groupby(
    'departamento'
).agg({

    'area':'mean',
    'numero_pisos':'mean',
    'estrato':'mean'

}).round(2)

indicadores_departamento


In [ ]:

# Porcentaje de registros por estrato

porcentaje_estrato = (
    df['estrato']
    .value_counts(normalize=True) * 100
).round(2)

porcentaje_estrato



# 2.4 Elaboración de tablas y gráficos comparativos

En esta sección se generan visualizaciones comparativas para analizar diferencias territoriales y comportamiento de las variables.


In [ ]:

# Tabla cruzada departamento vs estrato

tabla_departamentos = pd.crosstab(
    df['departamento'],
    df['estrato']
)

tabla_departamentos


In [ ]:

# Heatmap de estratos por departamento

plt.figure(figsize=(12,6))

sns.heatmap(
    tabla_departamentos,
    annot=True,
    fmt='d'
)

plt.title('Distribución de estratos por departamento')

plt.show()


In [ ]:

# Distribución del área por estrato

plt.figure(figsize=(10,5))

sns.boxplot(
    x='estrato',
    y='area',
    data=df
)

plt.title('Distribución del área por estrato')

plt.show()


In [ ]:

# Área promedio por departamento

df.groupby(
    'departamento'
)['area'].mean().sort_values().plot(
    kind='barh',
    figsize=(10,8)
)

plt.title('Área promedio por departamento')

plt.xlabel('Área promedio')

plt.show()



# 2.5 Modelo descriptivo de comparación territorial entre departamentos

Se realiza una comparación descriptiva entre departamentos seleccionados para identificar diferencias en variables socioeconómicas y físicas.


In [ ]:

# Selección de departamentos de análisis

df_modelo = df[
    df['departamento'].isin([
        'Cauca',
        'Antioquia',
        'Cundinamarca'
    ])
]

df_modelo.head()


In [ ]:

# Comparación territorial

comparacion = df_modelo.groupby(
    'departamento'
).agg({

    'area':'mean',
    'numero_pisos':'mean',
    'estrato':'mean'

}).round(2)

comparacion


In [ ]:

# Visualización comparativa

comparacion.plot(
    kind='bar',
    figsize=(10,5)
)

plt.title('Comparación territorial entre departamentos')

plt.ylabel('Promedio')

plt.show()


In [ ]:

# Distribución territorial de estratos

sns.countplot(
    data=df_modelo,
    x='estrato',
    hue='departamento'
)

plt.title('Distribución territorial de estratos')

plt.show()



# 2.6 Implementación de dashboards

Se implementan dashboards interactivos para facilitar la visualización y exploración de los resultados descriptivos.


In [ ]:

# Dashboard interactivo de estratos por departamento

fig = px.histogram(
    df_modelo,
    x='estrato',
    color='departamento',
    barmode='group',
    title='Distribución de estratos por departamento'
)

fig.show()


In [ ]:

# Dashboard de área por estrato

fig = px.box(
    df_modelo,
    x='estrato',
    y='area',
    color='departamento',
    title='Área por estrato y departamento'
)

fig.show()


In [ ]:

# Dashboard territorial interactivo

fig = px.scatter(
    df_modelo,
    x='area',
    y='numero_pisos',
    color='departamento',
    size='estrato',
    hover_data=['grupo_estrato'],
    title='Relación entre área y número de pisos'
)

fig.show()



# Conclusiones preliminares

- Los estratos socioeconómicos presentan diferencias en área y cantidad de pisos.
- Los departamentos analizados muestran comportamientos territoriales distintos.
- La agrupación de estratos facilita el análisis descriptivo.
- Los indicadores construidos permiten identificar patrones socioeconómicos.
- Los dashboards interactivos facilitan la exploración visual de los resultados.
